# Python 함수 연결 예제

Python 함수가 Method로 등록되고, 선택한 데이터를 받아 결과를 내기까지의 과정을 살펴봅니다.

**함수 작성 → Method 등록 → 입력 선택 → 데이터 조회 → 결과 확인**

상품별 매출 순위로 기본 흐름을 보고, OLS 회귀분석으로 여러 변수를 연결하는 방법을 알아봅니다.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# examples/ is outside the installed decision_layer package.
repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "examples/methods").is_dir()
)
sys.path.insert(0, str(repo_root))

from decision_layer.methods import Column, DataInput, Scope, python_method
from importlib import reload
from examples.methods.authoring import customer_analysis

# Read updated example code when rerunning this setup cell.
practice_session = reload(customer_analysis).practice_session

## 1. 상품별 매출 순위

먼저 간단한 순위 계산으로 함수와 데이터를 연결하는 과정을 살펴봅니다.

### 계산 함수

`value` 열의 순위를 계산하고 `group` 열과 함께 반환합니다. 

In [2]:
def rank_values(data, descending=True):
    """값이 큰 순서로 순위를 매깁니다. 같은 값은 같은 순위로 표시합니다."""
    return data[["group", "value"]].assign(
        rank=data["value"].rank(method="min", ascending=not descending).astype(int)
    )


frame = pd.DataFrame({"group": ["A", "B", "C"], "value": [30, 20, 20]})
standalone = rank_values(frame)
standalone

,group,value,rank
0,A,30,1
1,B,20,2
2,C,20,2


### Method로 등록하기

`python_method(...)`로 함수에 이름과 입력 조건을 붙입니다.

- `name`, `version`: Method의 이름과 버전
- `DataInput`: 함수에 전달할 데이터
- `Column`: 데이터에 필요한 열. `measure`는 지표, `dimension`은 차원입니다.

`practice_session()`은 예제 데이터가 준비된 로컬 세션을 만들고, 
`session.register(...)`는 이 세션에 Method를 추가합니다.

In [3]:
rank_method = python_method(
    rank_values,
    name="my_team.rank",
    version="1.0.0",
    inputs={
        "data": DataInput(
            columns={
                "value": Column(source="measure"),
                "group": Column(source="dimension"),
            },
        ),
    },
)

session = await practice_session()
session.register(rank_method)
print("등록된 Method:", [method.name for method in session.list_methods()])

등록된 Method: ['my_team.rank']


### 입력 선택과 실행

상품별 매출 순위를 구하도록 실제 데이터를 연결합니다.

- `bindings`: `value`에 매출, `group`에 상품을 연결합니다.
- `params`: `descending=True`로 매출이 큰 순서로 정렬합니다.
- `scope`: 조회 기간과 날짜 기준을 지정합니다.

Decision Layer는 Semantic Layer에서 상품별 매출을 조회합니다. 
- 매출은 `value`, 상품은 `group` 열에 담아 DataFrame을 만들고, 
- `rank_values` 함수에 전달해 순위를 계산합니다.

같은 방식으로 AI에게 “상품별 매출 순위를 알려줘”라고 요청하면, 
- AI가 사용할 Method와 지표, 기간을 선택합니다. 모호한 내용은 실행 전에 사용자에게 확인합니다. 
- Decision Layer가 데이터를 조회하고 함수를 실행하면, AI는 결과를 읽고 설명합니다.

구체적인 MCP 도구와 호출 예시는 아래 [AI는 MCP로 어떻게 실행하나요?](#_3-ai는-mcp로-어떻게-실행하나요)에서 다룹니다.

In [4]:
trial = await session.run(
    "my_team.rank",
    bindings={
        "value": "cube://shop/sales/revenue",
        "group": "cube://shop/sales/product",
    },
    params={"descending": True},
    scope=Scope(
        date_range=("2026-09-01", "2026-09-30"),
        time_dimension="cube://shop/sales/date",
    ),
)

trial.result.primary.data

[{'group': 'A', 'value': 30.0, 'rank': 1},
 {'group': 'B', 'value': 20.0, 'rank': 2},
 {'group': 'C', 'value': 20.0, 'rank': 2}]

## 2. 여러 변수를 사용하는 OLS 분석

순위 계산은 연결 과정을 보여주기 위한 간단한 예제였습니다. 여러 변수와 분석 조건을 다루는 코드에서는 같은 방법을 재사용하는 일이 더 중요해집니다.

- 데이터 과학자가 검토한 코드를 한 번 등록합니다.
- 여러 사용자와 AI가 필요한 데이터를 연결해 같은 분석을 사용합니다.

이번에는 고객별 광고 노출 횟수와 구매 금액의 관계를 OLS 회귀분석으로 살펴봅니다.

### 계산 함수

광고를 많이 본 고객이 구매도 더 많이 하는지 살펴봅니다. 고객의 나이와 이전 구매 금액도 함께 고려합니다.

함수에는 데이터와 분석할 열 이름을 전달합니다.

- `data`: 고객별 데이터
- `outcome`: 구매 금액
- `treatment`: 광고 노출 횟수
- `confounders`: 나이, 이전 구매 금액

In [5]:
def ols(data, outcome, treatment, confounders):
    """다른 변수들을 함께 고려해 관심 변수와 결과의 관계를 OLS로 분석합니다."""
    import statsmodels.api as sm

    design = sm.add_constant(data[[treatment, *confounders]], has_constant="add")
    fit = sm.OLS(data[outcome], design).fit()
    return {
        "value": float(fit.params[treatment]),
        "n_observations": int(fit.nobs),
        "r_squared": float(fit.rsquared),
        "coefficients": {key: float(value) for key, value in fit.params.items()},
        "causal_claim": "not_established",
        "uncertainty": "not_reported",
    }

### 분석 조건 검사하기

계산 전에 데이터를 확인할 검사 함수를 등록할 수 있습니다. 아래 `check_ols`는 빠진 값이나 중복된 열 등을 검사합니다.

검사 함수는 선택 사항입니다. 운영에서는 다음과 같이 등록할 수 있습니다.

- `checks.py`에 검사 함수를 작성합니다.
- `method.py`에서 함수를 가져와 `checks=[check_ols]`로 등록합니다.

검사 함수는 계산 함수와 같은 DataFrame과 열 이름을 받습니다. Error가 발생하면 실패 이유를 기록하고, `ols`는 실행하지 않습니다.


In [6]:
def check_ols(data, outcome, treatment, confounders):
    """OLS 계산에 필요한 데이터와 변수 구성을 확인합니다."""
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm

    columns = [outcome, treatment, *confounders]
    if len(set(columns)) != len(columns):
        raise ValueError("같은 열을 두 번 선택했습니다.")

    values = data[columns]
    if not all(pd.api.types.is_numeric_dtype(values[col]) for col in columns):
        raise ValueError("숫자형 열만 선택해 주세요.")
    if not np.isfinite(values.to_numpy(dtype=float)).all():
        raise ValueError("결측값이나 무한대를 먼저 처리해 주세요.")
    if values[outcome].nunique() < 2:
        raise ValueError("결과 변수의 값이 모두 같습니다.")

    design = sm.add_constant(values[[treatment, *confounders]], has_constant="add")
    if len(data) <= design.shape[1]:
        raise ValueError(
            "분석할 변수가 많습니다. 데이터를 늘리거나 변수를 줄여 주세요."
        )
    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError(
            "설명변수들이 서로 완전히 겹칩니다. 변수 구성을 확인해 주세요."
        )

### 직접 실행하기


In [7]:
from examples.methods.authoring.customer_analysis import customer_sample

customers = customer_sample()

check_ols(
    customers,
    outcome="purchase_amount",
    treatment="ad_exposures",
    confounders=["age", "prior_purchase_amount"],
)

direct_ols = ols(
    customers,
    outcome="purchase_amount",
    treatment="ad_exposures",
    confounders=["age", "prior_purchase_amount"],
)

direct_ols

{'value': 50.00000000000206,
 'n_observations': 200,
 'r_squared': 1.0,
 'coefficients': {'const': 999.999999999618,
  'ad_exposures': 50.00000000000206,
  'age': 9.999999999999813,
  'prior_purchase_amount': 0.20000000000000567},
 'causal_claim': 'not_established',
 'uncertainty': 'not_reported'}

### Method로 등록하기

- `role="entity_key"`는 고객별 한 행의 기준입니다. 
- `multiple=True`로 여러 변수를 선택할 수 있습니다.
- `checks=[check_ols]`로 검사 함수를 연결합니다. 

Method를 실행하면 Decision Layer가 순서대로 처리합니다.

**데이터 조회 → 기본 입력 검사 → `check_ols` → 통과하면 `ols` 실행**

**기본 입력 검사**는 `DataInput`에 정한 조건을 확인하는 과정입니다.

- `minimum_rows=10`: 데이터가 10행 이상인지
- `columns`: 필요한 열이 모두 있는지
- `dtype="number"`: 값이 숫자이고, 결측값이나 무한대가 없는지
- `role="entity_key"`: 고객 ID가 비어 있거나 중복되지 않는지

하나라도 실패하면 이유를 기록하고 계산을 멈춥니다. 모두 통과하면 `check_ols`가 변수 간 겹침 등 OLS 계산에 필요한 조건을 추가로 검사합니다.


In [8]:
my_ols = python_method(
    ols,
    name="my_team.ols",
    version="1.0.0",
    inputs={
        "data": DataInput(
            columns={
                "entity_id": Column(source="dimension", role="entity_key"),
                "outcome": Column(source="measure", dtype="number"),
                "treatment": Column(
                    source=["dimension", "measure"],
                    dtype="number",
                ),
                "confounders": Column(
                    source=["dimension", "measure"],
                    dtype="number",
                    multiple=True,
                ),
            },
            minimum_rows=10,
        )
    },
    checks=[check_ols],
    requires_packages={"statsmodels": ">=0.14,<0.16"},
)

session.register(my_ols)
print("등록된 Method:", [method.name for method in session.list_methods()])

등록된 Method: ['my_team.rank', 'my_team.ols']


### 입력 선택과 실행


In [9]:
ols_trial = await session.run(
    "my_team.ols",
    bindings={
        "entity_id": "cube://shop/customers/id",
        "outcome": "cube://shop/customers/purchase_amount",
        "treatment": "cube://shop/customers/ad_exposures",
        "confounders": [
            "cube://shop/customers/age",
            "cube://shop/customers/prior_purchase_amount",
        ],
    },
    scope=Scope(
        date_range=("2026-02-01", "2026-02-28"),
        time_dimension="cube://shop/customers/date",
    ),
)

ols_trial.result.primary.data

{'value': 50.00000000000206,
 'n_observations': 200,
 'r_squared': 1.0,
 'coefficients': {'const': 999.999999999618,
  'treatment': 50.00000000000206,
  'confounders_0': 9.999999999999813,
  'confounders_1': 0.20000000000000567},
 'causal_claim': 'not_established',
 'uncertainty': 'not_reported'}

### 선택한 입력이 함수에 전달되기까지

**1. 입력 확인**

Decision Layer는 `bindings`로 선택한 항목이 Cube에 있는지 확인합니다. 각 항목의 종류와 자료형이 Method의 입력 조건에 맞는지도 검사합니다.

- `entity_id`: 고객 ID
- `outcome`: 구매 금액
- `treatment`: 광고 노출 횟수
- `confounders`: 나이, 이전 구매 금액

**2. Cube에 고객별 데이터 조회**

선택한 지표와 차원, 기간으로 Cube에 조회를 요청합니다. Cube는 모델에 정의된 집계와 조인을 적용해 고객별 데이터를 반환합니다.

<details>
<summary>Cube 조회</summary>

`/cubejs-api/v1/load`로 보내는 JSON입니다.

```json
{
  "query": {
    "dimensions": ["customers.id", "customers.age"],
    "measures": [
      "customers.purchase_amount",
      "customers.ad_exposures",
      "customers.prior_purchase_amount"
    ],
    "timeDimensions": [
      {
        "dimension": "customers.date",
        "dateRange": ["2026-02-01", "2026-02-28"]
      }
    ],
    "order": [["customers.id", "asc"]],
    "limit": 50000,
    "offset": 0
  }
}
```

</details>

**3. method에 데이터 전달**

조회 결과를 DataFrame으로 만들고, 열 이름을 Method에 등록한 이름으로 바꿉니다.

- 구매 금액 → `outcome`
- 광고 노출 횟수 → `treatment`
- 나이, 이전 구매 금액 → 각각 `confounders_0`, `confounders_1`

이 DataFrame과 열 이름으로 `ols`를 호출합니다.

```python
ols(
    data=조회한_DataFrame,
    outcome="outcome",
    treatment="treatment",
    confounders=["confounders_0", "confounders_1"],
)
```


## 3. AI는 MCP로 어떻게 실행하나요?

AI에게 “2026년 2월 고객의 광고 노출 횟수와 구매 금액의 관계를 나이와 이전 구매 금액을 고려해 분석해 줘”라고 요청한다면,

**`list_methods` → `describe_method` → 데이터 선택 → 실행 → 결과 확인** 순서로 살펴봅니다.

### 1. 등록된 Method를 확인합니다

`list_methods`로 이름과 설명을 확인합니다.

```python
await client.call_tool("list_methods", {})
```

```json
{
  "methods": [
    {
      "name": "my_team.rank",
      "description": "값이 큰 순서로 순위를 매깁니다. 같은 값은 같은 순위로 표시합니다."
    },
    {
      "name": "my_team.ols",
      "description": "다른 변수들을 함께 고려해 관심 변수와 결과의 관계를 OLS로 분석합니다."
    }
  ]
}
```

AI는 설명을 읽고 질문에 맞는 OLS를 선택합니다. 이 설명은 함수의 docstring에서 가져옵니다. 필요한 데이터와 패키지는 실행할 때 검사합니다.


### 2. 필요한 입력을 확인합니다

선택한 OLS Method의 상세 조건을 확인합니다.

```python
await client.call_tool("describe_method", {"name": "my_team.ols"})
```

```json
{
  "name": "my_team.ols",
  "description": "다른 변수들을 함께 고려해 관심 변수와 결과의 관계를 OLS로 분석합니다.",
  "roles": {
    "entity_id": {
      "kind": "entity",
      "required": true,
      "multiple": false
    },
    "outcome": {
      "kind": "measure",
      "data_type": "number",
      "required": true,
      "multiple": false
    },
    "treatment": {
      "kind": "dimension",
      "accepted_kinds": [
        "dimension",
        "measure"
      ],
      "data_type": "number",
      "required": true,
      "multiple": false
    },
    "confounders": {
      "kind": "dimension",
      "accepted_kinds": [
        "dimension",
        "measure"
      ],
      "data_type": "number",
      "required": true,
      "multiple": true
    }
  },
  "parameters": {}
}
```

AI는 이 조건을 보고 다음 입력을 준비합니다.

- `entity_id`: 고객을 구분할 키
- `outcome`: 숫자형 지표 하나
- `treatment`: 숫자형 지표 또는 차원 하나
- `confounders`: 숫자형 지표 또는 차원 여러 개

`required`는 필수 여부, `multiple`은 여러 항목을 선택할 수 있는지를 뜻합니다. `accepted_kinds`가 있으면 지표와 차원 중 허용된 종류를 선택할 수 있습니다.

`parameters`가 비어 있으므로 별도 계산 옵션은 필요하지 않습니다. 순위 함수라면 여기에 `descending`이 표시됩니다.

### 3. 실제 데이터를 선택합니다

`search_semantic(query="customers")`로 고객 관련 항목을 찾습니다.

```json
{
  "count": 6,
  "objects": [
    {
      "ref": "cube://shop/customers/ad_exposures",
      "kind": "measure",
      "title": "광고 노출 횟수"
    },
    {
      "ref": "cube://shop/customers/age",
      "kind": "dimension",
      "title": "나이"
    },
    {
      "ref": "cube://shop/customers/date",
      "kind": "time_dimension",
      "title": "집계 기준일"
    },
    {
      "ref": "cube://shop/customers/id",
      "kind": "dimension",
      "title": "고객 ID"
    },
    {
      "ref": "cube://shop/customers/prior_purchase_amount",
      "kind": "measure",
      "title": "이전 기간 구매 금액"
    },
    {
      "ref": "cube://shop/customers/purchase_amount",
      "kind": "measure",
      "title": "구매 금액",
      "entity": "cube://shop/customers/id",
      "time_dimension": "cube://shop/customers/date"
    }
  ]
}
```

AI는 이름(`title`)과 종류(`kind`)를 보고 구매 금액과 광고 노출 횟수를 찾고, 해당 식별자(`ref`)를 입력에 연결합니다. 구매 금액의 `entity`는 고객별 조회 기준, `time_dimension`은 기간을 적용할 날짜 기준입니다.

이 예제에서는 구매 금액을 `outcome`, 광고 노출 횟수를 `treatment`, 나이와 이전 구매 금액을 `confounders`로 선택합니다. 항목의 의미나 함께 고려할 변수가 불분명하면 사용자에게 확인합니다.

아래 코드는 연결된 MCP 클라이언트에서 사용하는 호출 예시입니다. 셀을 실행하면 함수만 정의하며, 서버에 요청하지 않습니다.

실제로 호출하려면 같은 Method를 서버 프로젝트에 등록하고 `shop` Cube를 연결해야 합니다. 노트북의 `session.register(...)`는 로컬 세션에만 등록합니다.

In [ ]:
async def inspect_with_mcp(client):
    await client.call_tool("list_methods", {})
    method = await client.call_tool("describe_method", {"name": "my_team.ols"})
    catalog = await client.call_tool("search_semantic", {"query": "customers"})
    return method.structured_content, catalog.structured_content

### 4. 선택한 입력으로 실행합니다

`start_analysis`에 질문과 기간을 전달하면 Run을 만들고 `run_id`를 반환합니다.

```json
{
  "run_id": "run_example",
  "status": "open",
  "scope": {
    "date_range": [
      "2026-02-01",
      "2026-02-28"
    ],
    "time_dimension": "cube://shop/customers/date"
  },
  "next": {
    "tools": [
      "run_step"
    ],
    "run_id": "run_example"
  }
}
```

AI는 이 `run_id`와 선택한 지표를 `run_step`에 전달합니다. 기간은 이미 Run에 저장되어 있으므로 다시 넣지 않습니다.

질문에서 확인할 내용을 `goals`에 적고, `goal_ids`로 이번 계산에 연결합니다.


In [ ]:
async def run_ols_with_mcp(client):
    started = await client.call_tool(
        "start_analysis",
        {
            "question": "2026년 2월 고객의 광고 노출 횟수와 구매 금액의 관계를 나이와 이전 구매 금액을 고려해 분석해 줘",
            "date_range": ["2026-02-01", "2026-02-28"],
            "time_dimension": "cube://shop/customers/date",
            "goals": [
                {
                    "id": "relationship",
                    "description": "나이와 이전 구매 금액을 고려한 광고 노출과 구매 금액의 관계",
                }
            ],
        },
    )
    return await client.call_tool(
        "run_step",
        {
            "run_id": started.structured_content["run_id"],
            "method": "my_team.ols",
            "bindings": {
                "entity_id": "cube://shop/customers/id",
                "outcome": "cube://shop/customers/purchase_amount",
                "treatment": "cube://shop/customers/ad_exposures",
                "confounders": [
                    "cube://shop/customers/age",
                    "cube://shop/customers/prior_purchase_amount",
                ],
            },
            "params": {},
            "purpose": "나이와 이전 구매 금액을 고려해 광고 노출 횟수와 구매 금액의 관계 확인",
            "goal_ids": ["relationship"],
        },
    )

`run_step`이 계산을 마치면 다음 결과를 반환합니다.

```json
{
  "run_id": "run_example",
  "step_id": "step_1",
  "status": "success",
  "primary": {
    "type": "estimate",
    "data": {
      "value": 50.0,
      "n_observations": 200,
      "causal_claim": "not_established",
      "uncertainty": "not_reported"
    }
  },
  "validation": [
    {
      "code": "MINIMUM_ROWS",
      "status": "pass"
    },
    {
      "code": "PYTHON_CHECK",
      "status": "pass"
    }
  ],
  "warnings": [
    "Input checks do not establish source completeness, metric equivalence or causal identification."
  ],
  "provenance": {
    "method": "method://my_team/ols@1.0.0",
    "queries": 1,
    "rows_read": 200
  }
}
```

AI는 계수(`value`)와 관측 수(`n_observations`)뿐 아니라 검사 결과와 경고도 읽습니다. 검사를 통과했어도 인과관계까지 입증한 것은 아니므로, **“광고 노출이 많을수록 구매 금액이 높은 연관성이 있다”**고 설명합니다.

- `running`: 계산 중이므로 `wait_for_run`으로 기다립니다.
- `needs_input`: 부족한 입력을 사용자에게 확인합니다.
- `refused`: 거절 이유를 설명합니다.

### 데이터 조회와 함수 호출은 누가 할까요?

이 Python Method에서 AI는 Cube JSON이나 DataFrame을 직접 보내지 않습니다. Decision Layer가 `bindings`와 기간을 확인해 Cube 쿼리를 만들고 `/cubejs-api/v1/load`로 요청합니다. 지표는 `measures`, 고객 ID와 나이는 `dimensions`, 기간은 `timeDimensions`에 들어갑니다.

아래 셀은 앞서 로컬에서 OLS를 실행할 때 실제로 생성한 Cube 조회 조건입니다. MCP의 `run_step`도 같은 조회 코드를 사용합니다.


In [ ]:
ols_trial.queries[0].native_query

{'dimensions': ['customers.id', 'customers.age'],
 'measures': ['customers.purchase_amount',
              'customers.ad_exposures',
              'customers.prior_purchase_amount'],
 'timeDimensions': [{'dimension': 'customers.date',
                     'dateRange': ['2026-02-01', '2026-02-28']}],
 'order': [['customers.id', 'asc']]}

Cube가 고객별 데이터를 반환하면 Decision Layer는 열 이름을 바꾼 뒤 **`DataInput` 조건 확인 → `check_ols` → `ols`** 순서로 실행합니다.

이 흐름은 Python Method의 예입니다. `SKILL.md`를 실행할 때는 AI가 공통 지침에 따라 `run_skill_query`로 조회하고 `finish_skill_step`으로 마무리합니다.


### 5. 기록을 확인하고 결론을 남깁니다

`get_run(run_id="run_example")`으로 실제 사용한 입력과 저장된 결과를 확인합니다.

```json
{
  "run_id": "run_example",
  "status": "open",
  "steps": [
    {
      "id": "step_1",
      "method": "my_team.ols",
      "bindings": {
        "entity_id": "cube://shop/customers/id",
        "outcome": "cube://shop/customers/purchase_amount",
        "treatment": "cube://shop/customers/ad_exposures",
        "confounders": [
          "cube://shop/customers/age",
          "cube://shop/customers/prior_purchase_amount"
        ]
      },
      "params": {},
      "result": {
        "status": "success",
        "primary": {
          "type": "estimate",
          "data": {
            "value": 50.0,
            "n_observations": 200
          }
        }
      }
    }
  ]
}
```

AI는 요청한 기간과 변수가 맞는지 확인한 뒤 결과를 설명합니다. Run의 `open`은 결론을 아직 기록하지 않았다는 뜻입니다. 계산 성공 여부는 각 단계의 `result.status`로 확인합니다.

AI가 결과를 바탕으로 작성한 결론을 `complete_run`에 전달하면, 다음과 같이 저장됩니다.

```json
{
  "run_id": "run_example",
  "status": "completed",
  "conclusion": {
    "source": "caller",
    "answer": "나이와 이전 구매 금액을 함께 고려했을 때, 광고 노출 1회 증가와 구매 금액 50 증가가 연관되어 있습니다.",
    "limitations": [
      "합성 데이터의 분석 결과입니다.",
      "광고의 인과 효과를 입증한 결과는 아닙니다."
    ]
  }
}
```

`conclusion`은 AI가 전달한 설명입니다. 서버가 새로운 분석을 수행하거나 이 문장의 타당성을 입증한 것은 아닙니다.

## 프로젝트에서 사용하기

노트북에서는 함수를 셀에 정의했습니다. 프로젝트에 저장할 때는 다음처럼 나눌 수 있습니다.

```text
methods/ols/
├── method.py        # 계산 함수와 python_method 등록
├── checks.py        # 선택 사항: 검사 함수
└── test_method.py   # 계산과 검사가 맞는지 확인하는 테스트
```

`checks.py`에 옮겼다면 `method.py`에서 `from .checks import check_ols`로 가져오고, 위와 같이 `checks=[check_ols]`에 전달합니다. 서버가 자동으로 읽는 파일은 `method.py`입니다.

`test_method.py`는 개발 중 `decision-layer test`로 실행합니다. 분석할 때마다 실행되는 검사 함수와는 역할이 다릅니다.

[프로젝트에 등록하기](../../ko/guides/method-projects.md)에서 파일 구성과 서버 실행 방법을 살펴봅니다.
